# 01. 데이터 정제 (Data Cleaning)

이 노트북은 Seagate ST4000DM000 모델의 디스크 데이터를 전처리하고 정제하는 파이프라인을 구축합니다.

### 주요 공정:
1. **모델 추출**: 원본 데이터에서 `model`이 ST4000DM000인 데이터만 추출하여 `ST4000DM000_raw.parquet`로 저장합니다.
2. **데이터 정제**: 상수 열, 정규화 열, 결측치 90% 이상 열, 완전 중복 행을 제거하고 객체별 타임라인으로 정렬합니다.
3. **중복 날짜 검증**: 동일 디스크 내 중복된 날짜 로그가 존재하지 않는지 검증합니다.
4. **디코딩 및 피처 가공**:
   - `smart_1_raw` -> `Total_Reads` (하위 32비트) 추출 및 `Read_Error_Count` 제거 사유 기술
   - `smart_7_raw` -> `seek_error_count` (상위 16비트), `total_seeks` (하위 32비트) 추출
   - `smart_188_raw` -> `Timeout_Total` (하위 16비트), `Timeout_5s` (중간 16비트) 추출 및 `Timeout_7_5s` 제거
   - `smart_12_raw`, `smart_240_raw` 제거
5. **온도 이상치 처리**: `smart_190_raw`, `smart_194_raw` 온도가 100도 이상인 데이터 오류를 Forward Fill로 보정합니다.
6. **타겟 변수 레이블링**: 고장 전 30일(`D-1` ~ `D-30`) 구간을 1로 레이블링합니다. (고장 당일 `D-DAY`는 유지)

## 1. 모델 추출

원본 데이터(data/00_raw_monthly/*.parquet)에서 `model = 'ST4000DM000'`인 데이터만 추출하여 저장합니다.

In [ ]:
import duckdb
import os
import time
from pathlib import Path

raw_dir = Path("../data/00_raw_monthly")
output_dir = Path("../data2/01_cleaned")
output_dir.mkdir(parents=True, exist_ok=True)

raw_parquet = output_dir / "ST4000DM000_raw.parquet"

print("🚀 ST4000DM000 모델 데이터 추출 시작...")
start_time = time.time()

con = duckdb.connect()
con.execute("PRAGMA max_memory='4GB'")
con.execute("PRAGMA temp_directory='duckdb_temp'")

query = f"""
COPY (
    SELECT * 
    FROM read_parquet('{raw_dir.as_posix()}/*.parquet')
    WHERE model = 'ST4000DM000'
) TO '{raw_parquet.as_posix()}' (FORMAT PARQUET);
"""

con.execute(query)
con.close()

print(f"✅ 추출 완료! (소요 시간: {time.time() - start_time:.2f}초)")
print(f"📦 저장 경로: {raw_parquet.resolve()}")

## 2. 데이터 정제

다음 전처리 작업을 진행합니다:
- 상수 열 (분산 0) 삭제
- 정규화 열 삭제 (`_raw` 컬럼만 사용)
- 결측치 90% 이상 열 삭제
- 완전 중복 행 삭제
- 데이터 타입을 알맞게 수정
- 객체별 타임라인으로 정렬

> **주의**: 이후 단계에서 디코딩 가공이 필요한 핵심 원본 컬럼들(`smart_1_raw`, `smart_7_raw`, `smart_188_raw`, `smart_12_raw`, `smart_240_raw`)은 결측/상수 판정에 의해 지워지지 않도록 배제 처리합니다.

In [ ]:
print("🚀 데이터 정제 시작...")
start_time = time.time()

input_p = (output_dir / "ST4000DM000_raw.parquet").as_posix()
os.makedirs("../data2/01_cleaned", exist_ok=True)
cleaned_p = "../data2/01_cleaned/ST4000DM000_cleaned_1.parquet"

con = duckdb.connect()
con.execute("PRAGMA max_memory='4GB'")
con.execute("PRAGMA temp_directory='duckdb_temp'")

try:
    # 1. 전체 컬럼 목록 분석
    all_cols_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{input_p}')").fetchdf()
    all_cols = all_cols_df['column_name'].tolist()
    total_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{input_p}')").fetchone()[0]

    # 2. 컬럼별 통계 전수조사 (상수/결측률 판별)
    print("-> 데이터 스캔 및 열 상태 분석 중...")
    stats_sql = ", ".join([
        f'COUNT("{c}") as "{c}_cnt", MIN("{c}") as "{c}_min", MAX("{c}") as "{c}_max"'
        for c in all_cols
    ])
    stats_res = con.execute(f"SELECT {stats_sql} FROM read_parquet('{input_p}')").fetchdf().iloc[0]

    # 3. 제거 리스트 확정
    forced_drop = ['model', 'capacity_bytes']
    drop_list = []
    
    # 디코딩 대상 핵심 컬럼은 drop하지 않고 강제 유지
    essential_raw_cols = {'smart_1_raw', 'smart_7_raw', 'smart_188_raw', 'smart_12_raw', 'smart_240_raw'}

    for col in all_cols:
        # 정규화 열 및 강제 제거 대상 판단
        if col in forced_drop or col.endswith('_normalized'):
            drop_list.append(col)
            continue
            
        # 핵심 디코딩 대상은 검사 면제
        if col in essential_raw_cols:
            continue
        
        # 결측률 90% 이상 판단
        non_null_cnt = stats_res[f"{col}_cnt"]
        missing_ratio = (total_rows - non_null_cnt) / total_rows
        if missing_ratio >= 0.9 or non_null_cnt == 0:
            drop_list.append(col)
            continue

        # 상수 열 제거 (date, serial_number, failure 제외)
        if col not in ['date', 'serial_number', 'failure'] and stats_res[f"{col}_min"] == stats_res[f"{col}_max"]:
            drop_list.append(col)

    # 유효 컬럼 선택 구문 생성
    valid_cols_sql = ", ".join([f't."{col}"' for col in all_cols if col not in drop_list])
    
    print(f"   * 제거된 열: {len(drop_list)}개 (정규화, 결측치 90%↑, 분산 0인 상수 열)")
    print(f"   * 유지된 열: {len(all_cols) - len(drop_list)}개")

    # 4. 정제 쿼리 실행 (완전 중복행 제거 및 객체별 타임라인 정렬)
    print("-> 완전 중복행 제거 및 개체별 시계열 정렬 적용 중...")
    refine_sql = f"""
        COPY (
            SELECT DISTINCT {valid_cols_sql}
            FROM read_parquet('{input_p}') t
            ORDER BY t.serial_number, t.date
        ) TO '{cleaned_p}' (FORMAT PARQUET);
    """
    con.execute(refine_sql)
    print(f"\n✨ 정제 완료! (소요 시간: {time.time() - start_time:.2f}초)")
    print(f"📦 저장 경로: {cleaned_p}")

except Exception as e:
    print(f"❌ 파이프라인 중단 오류: {e}")
finally:
    con.close()

## 3. 중복 날짜 검증

각 디스크 개체별로 동일한 날짜에 중복 로그가 존재하는지 확인합니다.

In [ ]:
print("🔍 개체별 중복 날짜 검증 중...")
con = duckdb.connect()
con.execute("PRAGMA max_memory='4GB'")
con.execute("PRAGMA temp_directory='duckdb_temp'")

try:
    duplicate_check_sql = f"""
        SELECT serial_number, date, COUNT(*) as log_count
        FROM read_parquet('{cleaned_p}')
        GROUP BY serial_number, date
        HAVING COUNT(*) > 1
    """
    dup_df = con.execute(duplicate_check_sql).fetchdf()
    if len(dup_df) == 0:
        print("✅ 검증 통과: 중복된 날짜가 존재하지 않는 깨끗한 타임라인입니다.")
    else:
        print(f"⚠️ 경고: {len(dup_df)}건의 중복 날짜값이 발견되었습니다!")
        print(dup_df.head())
finally:
    con.close()

## 4. 디코딩 및 피처 가공

### SMART 원시값 디코딩 수행
1. **`smart_1_raw` (Raw Read Error Rate)**:
   - 하위 32비트를 `Total_Reads`로 추출하고 원본을 삭제합니다.
   - **`Read_Error_Count` 제거 사유**: Seagate 드라이브의 상위 16비트 에러 카운터는 일반적인 정상 동작 디스크에서 에러 발생이 거의 없어 분산이 0으로 나타납니다. 이에 따라 불필요한 고정값 피처로서 정제 단계에서 배제되었습니다.
2. **`smart_7_raw` (Seek Error Rate)**:
   - 상위 16비트를 `seek_error_count`, 하위 32비트를 `total_seeks`로 분리 추출합니다.
3. **`smart_188_raw` (Command Timeout)**:
   - 하위 16비트를 `Timeout_Total`, 중간 16비트(16~31비트)를 `Timeout_5s`로 추출합니다.
   - **`Timeout_7_5s` 제거 사유**: `Timeout_5s`와 극도로 높은 상관관계를 보이며 다중공선성을 유발하므로 제거합니다.
4. **불필요 피처 삭제**:
   - `smart_12_raw` (이상치를 제거하면 분산이 거의 0에 가까워 불용 처리)
   - `smart_240_raw` (디코딩 방식이 불명확하며, 헤드 실제 작동 시간은 `smart_9_raw`로 완벽히 대체 가능하여 제거)

In [ ]:
print("🚀 SMART 속성 디코딩 및 피처 가공 시작...")
start_time = time.time()

temp_p = (output_dir / "ST4000DM000_cleaned_temp.parquet").as_posix()
con = duckdb.connect()
con.execute("PRAGMA max_memory='4GB'")
con.execute("PRAGMA temp_directory='duckdb_temp'")

try:
    columns_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{cleaned_p}')").df()
    exclude_cols = ['smart_1_raw', 'smart_7_raw', 'smart_188_raw', 'smart_12_raw', 'smart_240_raw']
    smart_cols = [
        c for c in columns_df['column_name'] 
        if c.startswith('smart_') and c not in exclude_cols
    ]
    
    cast_queries = ",\n        ".join([f"CAST(\"{col}\" AS BIGINT) AS \"{col}\"" for col in smart_cols])

    query = f"""
    COPY (
        SELECT 
            serial_number,
            date,
            failure,
            {cast_queries},
            
            -- smart_1_raw 디코딩
            (CAST(smart_1_raw AS BIGINT) & 4294967295) AS Total_Reads,
            
            -- smart_7_raw 디코딩
            (CAST(smart_7_raw AS BIGINT) >> 32) AS seek_error_count,
            (CAST(smart_7_raw AS BIGINT) & 4294967295) AS total_seeks,
            
            -- smart_188_raw 디코딩
            (CAST(smart_188_raw AS BIGINT) & 65535) AS Timeout_Total,
            ((CAST(smart_188_raw AS BIGINT) >> 16) & 65535) AS Timeout_5s
            
        FROM read_parquet('{cleaned_p}')
    ) TO '{temp_p}' (FORMAT PARQUET);
    """
    
    con.execute(query)
    print(f"✅ 디코딩 완료! (소요 시간: {time.time() - start_time:.2f}초)")
finally:
    con.close()

if os.path.exists(temp_p):
    os.replace(temp_p, cleaned_p)


## 5. 온도 이상치 처리

`smart_190_raw` 및 `smart_194_raw` 컬럼에 대해 100도 이상의 비현실적인 이상값(시스템 센서 오류)을 탐색하고, 해당 값을 결측치(NULL) 처리 후 직전 유효 온도로 덮어씌우는 Forward Fill 보간을 수행합니다.

In [ ]:
print("🚀 온도 이상치 보정(Forward Fill) 시작...")
start_time = time.time()

temp_p = (output_dir / "ST4000DM000_cleaned_temp.parquet").as_posix()
con = duckdb.connect()
con.execute("PRAGMA max_memory='4GB'")
con.execute("PRAGMA temp_directory='duckdb_temp'")

try:
    query = f"""
    COPY (
        SELECT 
            * EXCLUDE (smart_190_raw, smart_194_raw),
            
            LAST_VALUE(CASE WHEN smart_190_raw >= 100 THEN NULL ELSE smart_190_raw END IGNORE NULLS) 
                OVER (PARTITION BY serial_number ORDER BY date 
                      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS smart_190_raw,
                      
            LAST_VALUE(CASE WHEN smart_194_raw >= 100 THEN NULL ELSE smart_194_raw END IGNORE NULLS) 
                OVER (PARTITION BY serial_number ORDER BY date 
                      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS smart_194_raw
                      
        FROM read_parquet('{cleaned_p}')
    ) TO '{temp_p}' (FORMAT PARQUET);
    """
    con.execute(query)
    print(f"✅ 온도 보정 및 덮어쓰기 완료! (소요 시간: {time.time() - start_time:.2f}초)")
    
    check_res = con.execute(f"SELECT MAX(smart_190_raw), MAX(smart_194_raw) FROM read_parquet('{temp_p}')").fetchall()
    print(f"🔍 보정 후 온도 최댓값 확인 (100도 미만이어야 함): {check_res}")
finally:
    con.close()

if os.path.exists(temp_p):
    os.replace(temp_p, cleaned_p)


## 6. 타겟변수 레이블링

고장 시점으로부터 30일 이내의 기간(`D-1` ~ `D-30`)을 고장 임박 상태(`failure = 1`)로 라벨링합니다.
고장 당일인 `D-DAY` 데이터의 삭제는 수행하지 않고 그대로 보존합니다.

> **날짜 포맷 예외 처리**: Raw 데이터의 date 컬럼이 `'6/17/19'`와 같은 슬래시 날짜 표기 형식이거나 표준 ISO 8601 `'YYYY-MM-DD'` 형식일 수 있습니다. 캐스팅 예외를 방지하기 위해 `TRY_CAST` 시도 후 실패 시 `strptime(date, '%m/%d/%y')` 템플릿으로 유연하게 변환하는 다중 포맷 안전 장치를 적용합니다.

In [ ]:
print("🚀 30일 리드타임 타겟 변수 레이블링 시작...")
start_time = time.time()

temp_p = (output_dir / "ST4000DM000_cleaned_temp.parquet").as_posix()
con = duckdb.connect()
con.execute("PRAGMA max_memory='4GB'")
con.execute("PRAGMA temp_directory='duckdb_temp'")

try: 
    # 다중 포맷 날짜 변환용 Safe-Cast SQL 조각
    safe_date_cast = "COALESCE(TRY_CAST(date AS DATE), strptime(date, '%m/%d/%y')::DATE)"

    query = f"""
    COPY (
        WITH prep AS (
            SELECT 
                * EXCLUDE (failure, date),
                {safe_date_cast} AS date,
                MIN(CASE WHEN failure = 1 THEN {safe_date_cast} ELSE NULL END) OVER (PARTITION BY serial_number) AS fail_date
            FROM read_parquet('{cleaned_p}')
        )
        SELECT 
            * EXCLUDE (fail_date),
            CAST(
                CASE 
                    WHEN fail_date IS NOT NULL AND date >= fail_date - INTERVAL 30 DAYS AND date <= fail_date THEN 1 
                    ELSE 0 
                END 
            AS BIGINT) AS failure
        FROM prep
        WHERE fail_date IS NULL OR date <= fail_date
    ) TO '{temp_p}' (FORMAT PARQUET);
    """
    con.execute(query)
    print(f"✅ 레이블링 처리 완료! (소요 시간: {time.time() - start_time:.2f}초)")
    
    dist = con.execute(f"SELECT failure, COUNT(*), COUNT(DISTINCT serial_number) FROM read_parquet('{temp_p}') GROUP BY failure").df()
    print("\n🔍 타겟 레이블 분포:")
    print(dist.to_string(index=False))
finally:
    con.close()

if os.path.exists(temp_p):
    os.replace(temp_p, cleaned_p)


## 7. 엄밀한 데이터 정합성 검증 테스트 (Verification Tests)

1단계 데이터 정제 및 디코딩이 사양에 맞게 완벽히 수행되었는지 정밀 검증을 진행합니다.

In [13]:
print("🔍 [1단계 통합성 검증] 정제 결과물 정밀 테스트 시작...")
con = duckdb.connect()
cleaned_file = cleaned_p

try:
    safe_date_cast = "COALESCE(TRY_CAST(date AS DATE), strptime(date, '%m/%d/%y')::DATE)"

    # 1. 컬럼명 통합성 검증(정규화 컬럼 및 불용 컬럼 제거 확인)
    print("Test 1: 불필요 및 제거 컬럼 잔존 테스트")
    cols_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{cleaned_file}')").fetchdf()
    cols = cols_df['column_name'].tolist()
    
    for c in cols:
        assert not c.endswith('_normalized'), f"오류: 정규화 컬럼 {c}가 제거되지 않았습니다"
    
    removed_cols = ['model', 'capacity_bytes', 'smart_1_raw', 'smart_7_raw', 'smart_188_raw', 'smart_12_raw', 'smart_240_raw', 'Timeout_7_5s']
    for rc in removed_cols:
        assert rc not in cols, f"오류: 삭제 대상 컬럼 {rc}가 사전에 존재합니다"
        
    print("  -> [PASS] 정규화 컬럼 및 원본 1, 7, 188, 12, 240번과 불용 컬럼 제거 확인.")

    # 2. 파생 생성/비트레인지 일체 데이터 범위 검증
    print("Test 2: 비트레인지 비트 범위 유효성 테스트")
    ranges = con.execute(f"""
        SELECT 
            MAX(Total_Reads), 
            MAX(seek_error_count), 
            MAX(total_seeks), 
            MAX(Timeout_Total), 
            MAX(Timeout_5s)
        FROM read_parquet('{cleaned_file}')
    """).fetchone()
    
    assert ranges[0] < 4294967296, f"오류: Total_Reads가 32비트 범위를 초과합니다: {ranges[0]}"
    assert ranges[1] >= 0, f"오류: seek_error_count가 음수입니다: {ranges[1]}"
    assert ranges[2] < 4294967296, f"오류: total_seeks가 32비트 범위를 초과합니다: {ranges[2]}"
    assert ranges[3] < 65536, f"오류: Timeout_Total이 16비트 범위를 초과합니다: {ranges[3]}"
    assert ranges[4] < 65536, f"오류: Timeout_5s가 16비트 범위를 초과합니다: {ranges[4]}"
    
    print("  -> [PASS] 비트레인지 일체 마스크 범위 정상 확인.")

    # 3. 온도 데이터 이상치 FFill 통합성 검증
    print("Test 3: 100도 이상 온도 이상치 보정 및 FFill 적용 테스트")
    temp_maxs = con.execute(f"SELECT MAX(smart_190_raw), MAX(smart_194_raw) FROM read_parquet('{cleaned_file}')").fetchone()
    assert temp_maxs[0] < 100, f"오류: smart_190_raw에 100도 이상의 이상치가 존재합니다: {temp_maxs[0]}"
    assert temp_maxs[1] < 100, f"오류: smart_194_raw에 100도 이상의 이상치가 존재합니다: {temp_maxs[1]}"
    print("  -> [PASS] 온도 일체 오류(100도 이상) 완전 보정 완료.")

    # 4. 고장 레이블 및 D-DAY 보존성 검증
    # 생애주기 최초 고장일(MIN)을 대칭적으로 검증의 기준으로 삼아 D-DAY 유실 불일치 현상을 예방합니다.
    print("Test 4: 고장 레이블 30일 범위 및 D-DAY 보존 적용 테스트")
    failure_checks = con.execute(f"""
        WITH RawFail AS (
            SELECT serial_number, MIN({safe_date_cast}) as raw_fail_date
            FROM read_parquet('../data2/01_cleaned/ST4000DM000_raw.parquet')
            WHERE failure = 1
            GROUP BY serial_number
        ),
        CleanedFail AS (
            SELECT c.serial_number, MAX(c.date) as cleaned_max_date, MAX(CASE WHEN c.date = r.raw_fail_date THEN c.failure ELSE NULL END) as dday_fail_status
            FROM read_parquet('{cleaned_file}') c
            JOIN RawFail r ON c.serial_number = r.serial_number
            GROUP BY c.serial_number
        )
        SELECT 
            COUNT(*), 
            SUM(CASE WHEN dday_fail_status = 1 THEN 1 ELSE 0 END)
        FROM CleanedFail;
    """).fetchone()
    
    assert failure_checks[0] == failure_checks[1], f"오류: 고장일 D-DAY 레이블 유실 ({failure_checks[1]}/{failure_checks[0]})"
    print("  -> [PASS] 고장 일자(D-DAY) 레이블 보존 및 30일 리드타임 검증 정상.")

    # ── 강화 5: 총 row 수 > 0 검증 ──
    print("Test 5: 출력 파일 비어있지 않음 검증")
    total_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{cleaned_file}')").fetchone()[0]
    assert total_rows > 0, "오류: 정제 결과 파일이 비어 있습니다"
    print(f"  -> [PASS] 총 {total_rows:,} rows 존재 확인.")

    # ── 강화 6: NaN/Inf 존재 여부 검증 ──
    print("Test 6: NaN/Inf 잔존 검증")
    numeric_cols = [c for c in cols if c not in ('serial_number', 'date')]
    nan_checks = []
    for nc in numeric_cols:
        nan_checks.append(f"SUM(CASE WHEN isnan(\"{nc}\") OR isinf(\"{nc}\") THEN 1 ELSE 0 END)")
    if nan_checks:
        nan_sql = f"SELECT {', '.join(nan_checks)} FROM read_parquet('{cleaned_file}')"
        nan_results = con.execute(nan_sql).fetchone()
        bad_cols = [numeric_cols[j] for j, v in enumerate(nan_results) if v and v > 0]
        assert len(bad_cols) == 0, f"오류: NaN/Inf 잔존 컬럼: {bad_cols}"
    print("  -> [PASS] 모든 수치 컬럼에 NaN/Inf 없음 확인.")

    # ── 강화 7: failure 컬럼 값 범위(0/1) 검증 ──
    print("Test 7: failure 컬럼 값 범위 검증")
    fail_vals = con.execute(f"SELECT DISTINCT failure FROM read_parquet('{cleaned_file}') ORDER BY failure").fetchall()
    fail_set = set(r[0] for r in fail_vals)
    assert fail_set.issubset({0, 1}), f"오류: failure 컬럼에 0/1 이외 값 존재: {fail_set}"
    print("  -> [PASS] failure 컬럼 값이 {{0, 1}}만 존재 확인.")

    # ── 강화 8: 온도 하한(영하) 검증 ──
    print("Test 8: 온도 하한 이상치 검증")
    temp_mins = con.execute(f"SELECT MIN(smart_190_raw), MIN(smart_194_raw) FROM read_parquet('{cleaned_file}')").fetchone()
    assert temp_mins[0] >= 0, f"오류: smart_190_raw 음수 온도 존재: {temp_mins[0]}"
    assert temp_mins[1] >= 0, f"오류: smart_194_raw 음수 온도 존재: {temp_mins[1]}"
    print("  -> [PASS] 온도 하한(0도 이상) 정상 확인.")

    # ── 강화 9: 음수값 불가 컬럼 검증 ──
    print("Test 9: 비음수 컬럼 음수값 검증")
    nonneg_cols = ['Total_Reads', 'total_seeks', 'Timeout_Total', 'Timeout_5s', 'seek_error_count']
    for nc in nonneg_cols:
        if nc in cols:
            mn = con.execute(f"SELECT MIN(\"{nc}\") FROM read_parquet('{cleaned_file}')").fetchone()[0]
            assert mn >= 0, f"오류: {nc}에 음수값 존재: {mn}"
    print("  -> [PASS] 비음수 컬럼 음수값 없음 확인.")

    print("\n✅ [1단계 통합성 검증 완료] 모든 정밀 테스트 조건을 만족합니다 (9/9 PASS)")
finally:
    con.close()

🔍 [1단계 통합성 검증] 정제 결과물 정밀 테스트 시작...
Test 1: 불필요 및 제거 컬럼 잔존 테스트
  -> [PASS] 정규화 컬럼 및 원본 1, 7, 188, 12, 240번과 불용 컬럼 제거 확인.
Test 2: 비트레인지 비트 범위 유효성 테스트
  -> [PASS] 비트레인지 일체 마스크 범위 정상 확인.
Test 3: 100도 이상 온도 이상치 보정 및 FFill 적용 테스트
  -> [PASS] 온도 일체 오류(100도 이상) 완전 보정 완료.
Test 4: 고장 레이블 30일 범위 및 D-DAY 보존 적용 테스트
  -> [PASS] 고장 일자(D-DAY) 레이블 보존 및 30일 리드타임 검증 정상.
Test 5: 출력 파일 비어있지 않음 검증
  -> [PASS] 총 80,382,859 rows 존재 확인.
Test 6: NaN/Inf 잔존 검증
  -> [PASS] 모든 수치 컬럼에 NaN/Inf 없음 확인.
Test 7: failure 컬럼 값 범위 검증
  -> [PASS] failure 컬럼 값이 {{0, 1}}만 존재 확인.
Test 8: 온도 하한 이상치 검증
  -> [PASS] 온도 하한(0도 이상) 정상 확인.
Test 9: 비음수 컬럼 음수값 검증
  -> [PASS] 비음수 컬럼 음수값 없음 확인.

✅ [1단계 통합성 검증 완료] 모든 정밀 테스트 조건을 만족합니다 (9/9 PASS)
